# CausalPype Demo: U.S. Census Adult Income

**Dataset**: UCI Adult Income (1994 U.S. Census, ~32K records)

**Causal questions**:
- Does education causally increase income?
- Is the effect heterogeneous across demographics?
- Is there counterfactual unfairness by sex?

**Tasks demonstrated**: ATE, CATE, KNN Matching, Fairness Audit, Counterfactual, Validate, Pipeline + Report

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Download UCI Adult Income dataset directly from the repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]
raw = pd.read_csv(url, header=None, names=columns, na_values=" ?", skipinitialspace=True)
raw.dropna(inplace=True)
print(f"Loaded {len(raw)} records, {len(raw.columns)} columns")
raw.head()

Loaded 32561 records, 15 columns


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## Data Preparation

We encode the variables we need for causal analysis. Our DAG will use:
- **age**: exogenous
- **sex**: exogenous (binary: 1=Male, 0=Female)
- **education_num**: years of education (continuous)
- **hours_per_week**: work hours
- **high_income**: binary outcome (>50K)

In [2]:
# Encode variables for causal analysis
df = raw[["age", "sex", "education_num", "hours_per_week", "income"]].copy()
df["sex"] = (df["sex"] == "Male").astype(int)
df["high_income"] = (df["income"] == ">50K").astype(int)
df.drop(columns=["income"], inplace=True)

# Standardize continuous variables for better SCM fitting
df["age"] = (df["age"] - df["age"].mean()) / df["age"].std()
df["education_num"] = (df["education_num"] - df["education_num"].mean()) / df["education_num"].std()
df["hours_per_week"] = (df["hours_per_week"] - df["hours_per_week"].mean()) / df["hours_per_week"].std()

print(f"Shape: {df.shape}")
print(f"\nHigh income rate: {df['high_income'].mean():.1%}")
print(f"Male fraction: {df['sex'].mean():.1%}")
df.describe().round(2)

Shape: (32561, 5)

High income rate: 24.1%
Male fraction: 66.9%


,age,sex,education_num,hours_per_week,high_income
count,32561.00,32561.00,32561.00,32561.00,32561.00
mean,-0.00,0.67,0.00,-0.00,0.24
std,1.00,0.47,1.00,1.00,0.43
min,-1.58,0.00,-3.53,-3.19,0.00
25%,-0.78,0.00,-0.42,-0.04,0.00
50%,-0.12,1.00,-0.03,-0.04,0.00
75%,0.69,1.00,0.75,0.37,0.00
max,3.77,1.00,2.30,4.74,1.00


## Define Causal Graph & Fit Model

We specify our assumed DAG based on domain knowledge:
- **age** and **sex** are exogenous (root nodes)
- **age** influences education, work hours, and income
- **sex** influences education, work hours, and income
- **education** influences work hours and income
- **hours_per_week** influences income

In [3]:
import causalpype as cp

# Define DAG as adjacency dict
graph = {
    "age": ["education_num", "hours_per_week", "high_income"],
    "sex": ["education_num", "hours_per_week", "high_income"],
    "education_num": ["hours_per_week", "high_income"],
    "hours_per_week": ["high_income"],
}

# Use a subsample for speed (SCM fitting on 30K rows is slow)
df_sample = df.sample(n=5000, random_state=42)

model = cp.CausalModel(graph)
model.fit(df_sample)
print(model)

Fitting causal mechanism of node high_income: 100%|██████████| 5/5 [00:00<00:00, 179.81it/s]

CausalModel(fitted, nodes=['age', 'sex', 'education_num', 'hours_per_week', 'high_income'], edges=[('age', 'education_num'), ('age', 'hours_per_week'), ('age', 'high_income'), ('sex', 'education_num'), ('sex', 'hours_per_week'), ('sex', 'high_income'), ('education_num', 'hours_per_week'), ('education_num', 'high_income'), ('hours_per_week', 'high_income')])


## Task 1: Average Treatment Effect (ATE)

Does education causally increase the probability of high income?

We compare do(education_num = +1 SD) vs do(education_num = -1 SD).

In [4]:
# ATE: Effect of education on high income
ate_result = model.run(cp.ATE(
    treatment="education_num",
    outcome="high_income",
    treatment_value=1.0,   # +1 SD education
    control_value=-1.0,    # -1 SD education
    num_samples=3000,
))
print(ate_result)

╭────────────────────── ATE ──────────────────────╮
│                                                 │
│  Estimate: ↑ 0.3053                             │
│   Treatment                     education_num   │
│   Outcome                       high_income     │
│   Treatment Value               ↑ 1.0000        │
│   Control Value                 ↓ -1.0000       │
│   Num Samples                   3,000           │
│                                                 │
╰─────────────────────────────────────────────────╯

## Task 2: Conditional Average Treatment Effect (CATE)

Is the effect of education on income different for different age groups? We use EconML's LinearDML with automatic confounder adjustment.

In [5]:
# CATE: Heterogeneous effects of education, moderated by age
cate_result = model.run(cp.CATE(
    treatment="education_num",
    outcome="high_income",
    effect_modifiers=["age"],
    method="linear_dml",
))
print(cate_result)

╭──────────────────────── CATE ────────────────────────╮
│                                                      │
│  Estimate: ↑ 0.1221                                  │
│   Treatment                     education_num        │
│   Outcome                       high_income          │
│   Effect Modifiers              age                  │
│   Method                        linear_dml           │
│   Mean Effect                   ↑ 0.1221             │
│   STD Effect                    ↑ 0.0168             │
│   Bounds                        ↑ 0.0954, ↑ 0.1848   │
│                                                      │
╰──────────────────────────────────────────────────────╯

## Task 3: KNN Matching

A non-parametric estimate of the effect of sex on income using nearest-neighbor matching on covariates.

In [6]:
# KNN Matching: Effect of sex on income, matching on age and education
knn_result = model.run(cp.KNNIntervention(
    treatment="sex",
    outcome="high_income",
    k=10,
    match_on=["age", "education_num", "hours_per_week"],
    treatment_value=1,   # Male
    control_value=0,     # Female
))
print(knn_result)

╭────────────────────────── KNN Intervention ──────────────────────────╮
│                                                                      │
│  Estimate: ↑ 0.1519                                                  │
│   Treatment                     sex                                  │
│   Outcome                       high_income                          │
│   K                             10                                   │
│   Match On                      age, education_num, hours_per_week   │
│   ATE                           ↑ 0.1519                             │
│   ATT                           ↑ 0.1674                             │
│   ATC                           ↑ 0.1198                             │
│   STD ITE                       ↑ 0.4004                             │
│   Match Quality Treated         ↑ 0.3153                             │
│   Match Quality Control         ↑ 0.1994                             │
│   N Treated                     3,380                                │
│   N Control                     1,620                                │
│                                                                      │
╰──────────────────────────────────────────────────────────────────────╯

## Task 4: Counterfactual Analysis

For each observed individual: what would their income have been if they had +1 SD more education?

In [7]:
# Counterfactual: What if everyone had +1 SD education?
cf_result = model.run(cp.Counterfactual(
    interventions={"education_num": 1.0},
    outcome="high_income",
))
print(cf_result)

╭────────────────────── Counterfactual ──────────────────────╮
│                                                            │
│  Estimate: ↑ 0.4394                                        │
│   N Units                       5,000                      │
│   Outcome                       high_income                │
│   Factual Mean                  ↑ 0.2456                   │
│   Counterfactual Mean           ↑ 0.4394                   │
│   Mean Effect                   ↑ 0.1938                   │
│                                                            │
│  Interventions                                             │
│   education_num           ↑ 1.0000  ████████████████████   │
│                                                            │
╰────────────────────────────────────────────────────────────╯

## Task 5: Fairness Audit

Is there counterfactual unfairness by sex? If we intervene on sex (Male vs Female) while keeping everything else at its natural value, does income change?

In [8]:
# Fairness Audit: counterfactual disparity by sex
fairness_result = model.run(cp.FairnessAudit(
    protected_attribute="sex",
    outcome="high_income",
    privileged_value=1,      # Male
    unprivileged_value=0,    # Female
))
print(fairness_result)

╭─────────────── Fairness Audit ────────────────╮
│                                               │
│  Estimate: ↑ 0.1716                           │
│   Protected Attribute           sex           │
│   Outcome                       high_income   │
│   Counterfactual Disparity      ↑ 0.1716      │
│   Observational Gap             ↑ 0.2017      │
│   Mean Individual Unfairness    ↑ 0.1716      │
│   Max Individual Unfairness     ↑ 1.0000      │
│   N Privileged                  3,380         │
│   N Unprivileged                1,620         │
│                                               │
╰───────────────────────────────────────────────╯

## Task 6: Mediation Analysis

How much of the effect of sex on income goes through education vs. through direct discrimination?

In [9]:
# Mediation: sex -> [education_num, hours_per_week] -> high_income
mediation_result = model.run(cp.Mediation(
    treatment="sex",
    outcome="high_income",
    treatment_value=1,    # Male
    control_value=0,      # Female
))
print(mediation_result)

╭─────────────────────────── Mediation ───────────────────────────╮
│                                                                 │
│  Estimate: ↑ 0.0270                                             │
│   Treatment                     sex                             │
│   Outcome                       high_income                     │
│   Mediators                     education_num, hours_per_week   │
│   Total Effect                  ↑ 0.1716                        │
│   Natural Direct Effect         ↑ 0.1446                        │
│   Natural Indirect Effect       ↑ 0.0270                        │
│   Proportion Mediated           ↑ 0.1573                        │
│                                                                 │
╰─────────────────────────────────────────────────────────────────╯

## Task 7: Pipeline — Run Multiple Tasks at Once

CausalPype's pipeline lets you chain tasks and get a unified report.

In [10]:
# Run a full pipeline: ATE + Intervention + Fairness in one call
results = model.run([
    cp.ATE("education_num", "high_income", treatment_value=1.0, control_value=-1.0),
    cp.Intervention({"education_num": 1.5}, outcome="high_income"),
    cp.FairnessAudit("sex", "high_income"),
])

# Print the unified report
print(model.report())

╭────────────────────── ATE ──────────────────────╮
│                                                 │
│  Estimate: ↑ 0.3190                             │
│   Treatment                     education_num   │
│   Outcome                       high_income     │
│   Treatment Value               ↑ 1.0000        │
│   Control Value                 ↓ -1.0000       │
│   Num Samples                   2,000           │
│                                                 │
╰─────────────────────────────────────────────────╯

╭─────────────────────── Intervention ───────────────────────╮
│                                                            │
│  Estimate: ↑ 0.5525                                        │
│   Outcome                       high_income                │
│   Mean                          ↑ 0.5525                   │
│   STD                           ↑ 0.6569                   │
│                                                            │
│  Interventions                                             │
│   education_num           ↑ 1.5000  ████████████████████   │
│                                                            │
╰────────────────────────────────────────────────────────────╯

╭─────────────── Fairness Audit ────────────────╮
│                                               │
│  Estimate: ↑ 0.1716                           │
│   Protected Attribute           sex           │
│   Outcome                       high_income   │
│   Counterfactual Disparity      ↑ 0.1716      │
│   Observational Gap             ↑ 0.2017      │
│   Mean Individual Unfairness    ↑ 0.1716      │
│   Max Individual Unfairness     ↑ 1.0000      │
│   N Privileged                  3,380         │
│   N Unprivileged                1,620         │
│                                               │
╰───────────────────────────────────────────────╯

CausalPype Analysis Report

--- Causal Graph ---
  Nodes (5): ['age', 'sex', 'education_num', 'hours_per_week', 'high_income']
  Edges (9): [('age', 'education_num'), ('age', 'hours_per_week'), ('age', 'high_income'), ('sex', 'education_num'), ('sex', 'hours_per_week'), ('sex', 'high_income'), ('education_num', 'hours_per_week'), ('education_num', 'high_income'), ('hours_per_week', 'high_income')]
  Root nodes: ['age', 'sex']
  Leaf nodes: ['high_income']

--- Data ---
  Rows: 5000
  Columns: 5









In [11]:
# Also available as structured dict (e.g. for JSON export, dashboards, LLM consumption)
import json
report_dict = model.report(format="dict")
print(json.dumps(report_dict, indent=2, default=str))

{
  "graph": {
    "nodes": [
      "age",
      "sex",
      "education_num",
      "hours_per_week",
      "high_income"
    ],
    "edges": [
      [
        "age",
        "education_num"
      ],
      [
        "age",
        "hours_per_week"
      ],
      [
        "age",
        "high_income"
      ],
      [
        "sex",
        "education_num"
      ],
      [
        "sex",
        "hours_per_week"
      ],
      [
        "sex",
        "high_income"
      ],
      [
        "education_num",
        "hours_per_week"
      ],
      [
        "education_num",
        "high_income"
      ],
      [
        "hours_per_week",
        "high_income"
      ]
    ],
    "n_nodes": 5,
    "n_edges": 9
  },
  "data": {
    "n_rows": 5000,
    "columns": [
      "age",
      "sex",
      "education_num",
      "hours_per_week",
      "high_income"
    ]
  },
  "results": [
    {
      "task_name": "ATE",
      "estimate": 0.319,
      "details": {
        "treatment": "education_num